In [1]:
from pathlib import Path

note_files = sorted(Path("mt_onc_50").glob("*.txt"))
len(note_files), [file.name for file in note_files[:3]]

(50, ['mt_oncology_0.txt', 'mt_oncology_1.txt', 'mt_oncology_10.txt'])

In [2]:
import os
from pathlib import Path

spark_tmp = Path.home() / "spark-tmp"
spark_tmp.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(spark_tmp)
os.environ["SPARK_LOCAL_DIRS"] = str(spark_tmp)
os.environ["JAVA_TOOL_OPTIONS"] = f"-Djava.io.tmpdir={spark_tmp}"

In [3]:
from johnsnowlabs import nlp

spark = nlp.start()

rows = [
    (file.name, file.read_text(encoding="utf-8"))
    for file in note_files
]

notes_df = spark.createDataFrame(rows, ["file_name", "text"])

print("Number of notes in Spark:", notes_df.count())
notes_df.select("file_name").show(5, truncate=False)

📋 Loading license number 0 from /home/sengul/.johnsnowlabs/licenses/license_number_{number}_for_Spark-Healthcare.json


Picked up JAVA_TOOL_OPTIONS: -Djava.io.tmpdir=/home/sengul/spark-tmp
Picked up JAVA_TOOL_OPTIONS: -Djava.io.tmpdir=/home/sengul/spark-tmp
26/10/15 15:14:50 WARN Utils: Your hostname, Sengl resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/15 15:14:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/10/15 15:14:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


👌 Launched cpu optimized session with with: 🚀Spark-NLP==7.0.0, 💊Spark-Healthcare==7.0.0, running on ⚡ PySpark==3.5.0


Number of notes in Spark: 50
+------------------+
|file_name         |
+------------------+
|mt_oncology_0.txt |
|mt_oncology_1.txt |
|mt_oncology_10.txt|
|mt_oncology_11.txt|
|mt_oncology_12.txt|
+------------------+
only showing top 5 rows



In [4]:
sample_note = (
    notes_df
    .filter(notes_df.file_name == "mt_oncology_1.txt")
    .first()["text"]
)

print(sample_note[:2500])

Sample Type / Medical Specialty:
Hematology - Oncology
Sample Name:
BCCa Excision - Lower Lid
Description:
Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
(Medical Transcription Sample Report)
PREOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.
POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.
TITLE OF OPERATION:
Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
PROCEDURE:
The patient was brought into the operating room and prepped and draped in usual fashion. Xylocaine 2% with epinephrine was injected beneath the conjunctiva and skin of the lower lid and also beneath the conjunctiva and skin of the upper lid. A frontal nerve block was also given on the right upper lid. The anesthetic agent was also injected in th

In [5]:
document_assembler = nlp.DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = nlp.SentenceDetectorDLModel.pretrained(
    "sentence_detector_dl_healthcare", "en", "clinical/models"
) \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

sentence_pipeline = nlp.Pipeline(stages=[
    document_assembler,
    sentence_detector
])

sample_note = notes_df.filter(
    notes_df.file_name == "mt_oncology_1.txt"
)

sentences_df = sentence_pipeline.fit(sample_note).transform(sample_note)

sentences_df.selectExpr(
    "file_name",
    "explode(sentence) AS sentence"
).select(
    "file_name",
    "sentence.result"
).show(10, truncate=False)

sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[ | ]

26/10/15 15:15:11 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB


26/10/15 15:15:12 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


[ / ]Download done! Loading the resource.


2026-10-15 15:15:15.222305: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


[OK!]
+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file_name        |result                                                                                                                                                                             |
+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|mt_oncology_1.txt|Sample Type / Medical Specialty:\nHematology - Oncology                                                                                                                            |
|mt_oncology_1.txt|Sample Name:                                                                                                                                                                   

In [6]:
sentence_rows = sentences_df.selectExpr(
    "explode(sentence) AS sentence"
).select(
    "sentence.result"
)

print("Number of text segments:", sentence_rows.count())
sentence_rows.show(100, truncate=False)

Number of text segments: 27
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|result                                                                                                                                                                                                                                                                                                                   |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sample Type / Medical S

In [7]:
from pyspark.sql.functions import col

sentence_rows.filter(
    col("result").contains("Cryotherapy")
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|result                                                                                                                                                                                                                           |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Cryotherapy was then used to treat the nasal and temporal margins of the area of excision because of positive margins, and following this an antibiotic steroid ointment was instilled and a light pressure dressing was applied.|
+---------------------------------------------------------------------------------------

In [8]:
from johnsnowlabs import medical

tokenizer = nlp.Tokenizer() \
    .setInputCols(["sentence"]) \
    .setOutputCol("token")

word_embeddings = nlp.WordEmbeddingsModel.pretrained(
    "embeddings_clinical", "en", "clinical/models"
) \
    .setInputCols(["sentence", "token"]) \
    .setOutputCol("embeddings")

oncology_ner = medical.NerModel.pretrained(
    "ner_oncology", "en", "clinical/models"
) \
    .setInputCols(["sentence", "token", "embeddings"]) \
    .setOutputCol("ner")

ner_converter = medical.NerConverterInternal() \
    .setInputCols(["sentence", "token", "ner"]) \
    .setOutputCol("ner_chunk")

ner_pipeline = nlp.Pipeline(stages=[
    tokenizer,
    word_embeddings,
    oncology_ner,
    ner_converter
])

ner_output = ner_pipeline.fit(sentences_df).transform(sentences_df)

ner_output.selectExpr(
    "explode(ner_chunk) AS entity"
).select(
    "entity.result",
    "entity.metadata"
).show(50, truncate=False)

embeddings_clinical download started this may take some time.


26/10/15 15:15:36 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 1.6 GB
[ | ]

26/10/15 15:15:37 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB


...

26/10/15 15:15:52 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Download done! Loading the resource.
[ — ]

2026-10-15 15:15:53.898006: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_name_='' resource_device_name_='/device:GPU:0' supported_device_types_=[CPU] possible_devices_=[]
AssignSub: CPU 
ScatterAdd: CPU 
NoOp: CPU 
Sqrt: CPU 
UnsortedSegmentSum: CPU 
RealDiv: CPU 
Const: CPU 
StridedSlice: CPU 
RandomUniform: CPU 
AddV2: CPU 
Unique: CPU 
Mul: CPU 
VariableV2: CPU 
Assign: CPU 
Shape: CPU 
Identity: CPU 
Cast: CPU 
Sub: CPU 
GatherV2: CPU

[OK!]


+--------------------------+----------------------------------------------------------------------------------------------------------------+
|result                    |metadata                                                                                                        |
+--------------------------+----------------------------------------------------------------------------------------------------------------+
|Excision                  |{entity -> Cancer_Surgery, confidence -> 0.8851, ner_source -> ner_chunk, chunk -> 0, sentence -> 4}            |
|large basal cell          |{entity -> Histological_Type, confidence -> 0.7180667, ner_source -> ner_chunk, chunk -> 1, sentence -> 4}      |
|carcinoma                 |{entity -> Cancer_Dx, confidence -> 0.9946, ner_source -> ner_chunk, chunk -> 2, sentence -> 4}                 |
|right                     |{entity -> Direction, confidence -> 0.9943, ner_source -> ner_chunk, chunk -> 3, sentence -> 4}                 |
|lower

In [9]:
from sparknlp.pretrained import PretrainedPipeline

qa_pipeline = PretrainedPipeline(
    "clinical_notes_qa_base_onnx_pipeline",
    "en",
    "clinical/models"
)

question = "What was the postoperative diagnosis?"
context = sample_note.select("text").first()["text"]

qa_result = qa_pipeline.fullAnnotate([question], [context])
qa_result

clinical_notes_qa_base_onnx_pipeline download started this may take some time.


26/10/15 15:16:06 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approx size to download 971.9 MB
[ | ]

26/10/15 15:16:06 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


clinical_notes_qa_base_onnx_pipeline download started this may take some time.
Approximate size to download 971.9 MB


...

Download done! Loading the resource.
[ \ ]Using CPUs
[ / ]Using CPUs


26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:21 WARN OnnxTensor:

[OK!]


26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.            
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN OnnxTensor: Closing an already closed tensor.
26/10/15 15:16:27 WARN

[{'document_question': [Annotation(document, 0, 36, What was the postoperative diagnosis?, {'sentence': '0'}, [])],
  'document_context': [Annotation(document, 0, 3240, Sample Type / Medical Specialty:
   Hematology - Oncology
   Sample Name:
   BCCa Excision - Lower Lid
   Description:
   Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
   (Medical Transcription Sample Report)
   PREOPERATIVE DIAGNOSIS:
   Extremely large basal cell carcinoma, right lower lid.
   POSTOPERATIVE DIAGNOSIS:
   Extremely large basal cell carcinoma, right lower lid.
   TITLE OF OPERATION:
   Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
   PROCEDURE:
   The patient was brought into the operating room and prepped and draped in usual fashion. Xylocaine 2% with epinephrine was injected bene

In [10]:
start = context.find("POSTOPERATIVE DIAGNOSIS:")
end = context.find("TITLE OF OPERATION:", start)

evidence = context[start:end].strip()
print(evidence)

POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.


In [11]:
import re

answer_text = qa_result[0]["answer"][0].result

sections = [
    section.strip()
    for section in re.split(r"(?m)(?=^[A-Z][A-Z /-]+:\s*$)", context)
    if section.strip()
]

stop_words = {"what", "was", "the", "is", "a", "an", "of", "for", "to", "and", "in"}

query_terms = set(
    re.findall(r"[a-z0-9]+", f"{question} {answer_text}".lower())
) - stop_words

def overlap_score(section):
    section_terms = set(re.findall(r"[a-z0-9]+", section.lower()))
    return len(query_terms & section_terms)

evidence = max(sections, key=overlap_score)

print("Answer:", answer_text)
print("Evidence from the note:", evidence)

Answer: The postoperative diagnosis was extremely large basal cell carcinoma, right lower lid.
Evidence from the note: POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.


In [12]:
import re
import html
from IPython.display import display, HTML

# Extract the section heading and evidence sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(r"(?<=[.!?])\s+|\n+", evidence_body)[0].strip()

# Find the evidence sentence within the correct section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

# Split the note into sentences and locate the evidence sentence
matches = list(re.finditer(r"[^.!?\n]+[.!?]?", context))
sentences = [match.group().strip() for match in matches]

center = next(
    i for i, match in enumerate(matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to three sentences before and after the evidence
start = max(0, center - 3)
end = min(len(sentences), center + 4)
nearby_sentences = sentences[start:end]

# Highlight the evidence sentence
paragraph = " ".join(
    f"<mark style='background-color:#ffe066; padding:2px'>{html.escape(sentence)}</mark>"
    if evidence_sentence in sentence
    else html.escape(sentence)
    for sentence in nearby_sentences
)

display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p><b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))

In [13]:
import re
import html
from IPython.display import display, HTML

# Get the evidence section heading and sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(r"(?<=[.!?])\s+|\n+", evidence_body)[0].strip()

# Locate the evidence sentence inside its section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

# Split the note into sentence fragments and remove standalone headings
sentence_matches = []

for match in re.finditer(r"[^.!?\n]+[.!?]?", context):
    sentence = match.group().strip()

    is_heading = re.fullmatch(r"[A-Z][A-Z0-9 /_-]*:", sentence)

    if sentence and not is_heading:
        sentence_matches.append((match, sentence))

# Find the exact occurrence selected as evidence
center = next(
    i for i, (match, _) in enumerate(sentence_matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to five sentences before and after the evidence
start = max(0, center - 5)
end = min(len(sentence_matches), center + 6)

# Highlight only the selected evidence occurrence
formatted_sentences = []

for i, (_, sentence) in enumerate(sentence_matches[start:end], start=start):
    escaped_sentence = html.escape(sentence)

    if i == center:
        escaped_sentence = (
            f"<mark style='background-color:#ffe066; padding:2px'>"
            f"{escaped_sentence}</mark>"
        )

    formatted_sentences.append(escaped_sentence)

paragraph = " ".join(formatted_sentences)

display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p style='max-width:850px; line-height:1.7'>"
    f"<b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))

In [14]:
import re
import html
from IPython.display import display, HTML

# Get the evidence section heading and sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(
    r"(?<=[.!?])\s+|\n+",
    evidence_body
)[0].strip()

# Find the evidence sentence within the selected section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

if section_start == -1 or evidence_position == -1:
    raise ValueError("Could not locate the evidence in the source note.")

# Split the source note into sentence fragments
sentence_matches = []

for match in re.finditer(r"[^.!?\n]+[.!?]?", context):
    sentence = match.group().strip()

    # Skip standalone section headings
    is_heading = re.fullmatch(r"[A-Z][A-Z0-9 /_-]*:", sentence)

    if sentence and not is_heading:
        sentence_matches.append((match, sentence))

# Find the exact sentence occurrence selected as evidence
center = next(
    i
    for i, (match, _) in enumerate(sentence_matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to five sentences before and after the evidence
start = max(0, center - 5)
end = min(len(sentence_matches), center + 6)

# Highlight only the selected evidence sentence
formatted_sentences = []

for i, (_, sentence) in enumerate(sentence_matches[start:end], start=start):
    escaped_sentence = html.escape(sentence)

    if i == center:
        escaped_sentence = (
            f"<mark style='background-color:#ffe066; padding:2px'>"
            f"{escaped_sentence}</mark>"
        )

    formatted_sentences.append(escaped_sentence)

paragraph = " ".join(formatted_sentences)

# Display the answer, evidence section, and surrounding source text
display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p><b>Evidence section:</b> {html.escape(heading)}</p>"
    f"<p style='max-width:850px; line-height:1.7'>"
    f"<b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))